# Storico allenatori

Versione interattiva di `scripts/generate_coaches_history.py`. Per ogni coach stampa il riepilogo e visualizza il grafico nel notebook, senza salvare CSV o PNG.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_results
from src.statistics import count_results

## Funzioni di supporto

In [ ]:
def build_summary(coach: str, coach_results: pd.Series) -> pd.DataFrame:
    base_counts = count_results(coach_results)
    counts = {
        "Partecipazioni": base_counts["Partecipazioni"],
        "Vittorie": base_counts["Vittorie"],
        "Primi 3": base_counts["Primi 3"],
        "Risultati neutri": int((coach_results == "N").sum()),
        "Retrocessioni": base_counts["Retrocessioni"],
    }
    participations = counts["Partecipazioni"]
    percentages = {
        column: f"{(value / participations * 100 if participations else 0.0):.2f}%"
        for column, value in counts.items()
        if column != "Partecipazioni"
    }
    percentages["Partecipazioni"] = ""
    return pd.DataFrame(
        [
            {"Tipo": "Conteggi", "Nome": coach, **counts},
            {
                "Tipo": "Percentuali",
                "Nome": "",
                **{column: percentages[column] for column in counts},
            },
        ]
    )


def show_chart(coach: str, coach_results: pd.Series, conversion: dict[str, float]) -> None:
    seasons = [f"{year}/{str(year + 1)[-2:]}" for year in coach_results.index]
    values = [np.nan if result == "A" else conversion[result] for result in coach_results]
    x_positions = np.arange(len(seasons))

    figure, axis = plt.subplots(figsize=(9, 5))
    axis.plot(x_positions, values, marker="o", linewidth=2)
    axis.set_title(f"Risultati per stagione - {coach}")
    axis.set_xlabel("Stagione")
    axis.set_ylabel("Valore del risultato")
    axis.set_xticks(x_positions, seasons)
    axis.set_yticks(sorted(set(conversion.values())))
    axis.grid(True, alpha=0.3)
    figure.tight_layout()
    plt.show()
    plt.close(figure)

## Configurazione e dati

In [ ]:
with (PROJECT_ROOT / "configs" / "config.yaml").open(encoding="utf-8") as stream:
    config = yaml.safe_load(stream)

years = config["year_selection"]
conversion = config["result_to_number_conversion"]
results = load_results(
    PROJECT_ROOT / config["data"]["file_path"],
    years["first_year"],
    years["last_year"],
)

print(f"Stagioni caricate: {years['first_year']}/{str(years['first_year'] + 1)[-2:]} - {years['last_year']}/{str(years['last_year'] + 1)[-2:]}")
print(f"Allenatori: {len(results.columns)}")

## Riepiloghi e grafici

Per ispezionare un solo coach, sostituisci `results.columns` con una lista come `["Nome coach"]`.

In [ ]:
for coach in results.columns:
    print(f"\n{'=' * 80}\n{coach}\n{'=' * 80}")
    summary = build_summary(coach, results[coach])
    print(summary.to_string(index=False))
    show_chart(coach, results[coach], conversion)